# Module 2: SQL Analysis — Business Intelligence Queries

**Objective:** Load cleaned data into SQLite, then answer real business questions using SQL.

**Tools:** SQLite3, Pandas

---

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2.1 Load CSVs into SQLite

In [2]:
conn = sqlite3.connect(":memory:")

tables = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table_name, csv_file in tables.items():
    df = pd.read_csv(DATA_DIR / csv_file)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")

def run_query(sql: str) -> pd.DataFrame:
    """Execute a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn)

Loaded customers: 99,441 rows


Loaded orders: 99,441 rows


Loaded order_items: 112,650 rows
Loaded payments: 103,886 rows


Loaded reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded category_translation: 71 rows


## 2.2 Business Question 1: Monthly Revenue Trend

> *"What is the monthly revenue trend? Are we growing?"*

In [3]:
q1 = run_query("""
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) AS month,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(p.payment_value), 2) AS total_revenue,
        ROUND(AVG(p.payment_value), 2) AS avg_order_value
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
    ORDER BY month
""")

print("Monthly Revenue Trend (delivered orders):")
display(q1)

Monthly Revenue Trend (delivered orders):


,month,total_orders,total_revenue,avg_order_value
0,2016-10,265,46566.71,165.13
1,2016-12,1,19.62,19.62
2,2017-01,750,127545.67,159.63
3,2017-02,1653,271298.65,155.12
4,2017-03,2546,414369.39,153.47
5,2017-04,2303,390952.18,160.49
6,2017-05,3546,567066.73,149.74
7,2017-06,3135,490225.60,147.53
8,2017-07,3872,566403.93,136.58
9,2017-08,4193,646000.61,147.05


## 2.3 Business Question 2: Top 10 Product Categories by Revenue

> *"Which product categories generate the most revenue?"*

In [4]:
q2 = run_query("""
    SELECT
        COALESCE(ct.product_category_name_english, 'other') AS category,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(oi.price), 2) AS total_revenue,
        ROUND(AVG(oi.price), 2) AS avg_price,
        ROUND(SUM(oi.freight_value), 2) AS total_freight
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products pr ON oi.product_id = pr.product_id
    LEFT JOIN category_translation ct ON pr.product_category_name = ct.product_category_name
    WHERE o.order_status = 'delivered'
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print("Top 10 Product Categories by Revenue:")
display(q2)

Top 10 Product Categories by Revenue:


,category,total_orders,total_revenue,avg_price,total_freight
0,health_beauty,8647,1233131.72,130.28,178957.81
1,watches_gifts,5495,1166176.98,199.04,98156.14
2,bed_bath_table,9272,1023434.76,93.44,201774.50
3,sports_leisure,7530,954852.55,113.25,163404.36
4,computers_accessories,6530,888724.61,116.26,143999.16
5,furniture_decor,6307,711927.69,87.25,168402.23
6,housewares,5743,615628.69,90.60,142763.56
7,cool_stuff,3559,610204.10,164.12,81476.79
8,auto,3810,578966.65,139.85,90488.10
9,toys,3804,471286.48,116.94,75774.58


## 2.4 Business Question 3: Customer Lifetime Value (Top Spenders)

> *"Who are our most valuable customers?"*

In [5]:
q3 = run_query("""
    SELECT
        c.customer_unique_id,
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(p.payment_value), 2) AS total_spent,
        ROUND(AVG(p.payment_value), 2) AS avg_order_value,
        MIN(o.order_purchase_timestamp) AS first_order,
        MAX(o.order_purchase_timestamp) AS last_order
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
    ORDER BY total_spent DESC
    LIMIT 15
""")

print("Top 15 Customers by Lifetime Value:")
display(q3)

Top 15 Customers by Lifetime Value:


,customer_unique_id,customer_state,total_orders,total_spent,avg_order_value,first_order,last_order
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,13664.08,2017-09-29 15:24:52,2017-09-29 15:24:52
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,3785.82,2017-04-01 15:58:40,2017-04-01 15:58:41
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,7274.88,2018-07-15 14:49:44,2018-07-15 14:49:44
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,6929.31,2017-02-12 20:37:36,2017-02-12 20:37:36
4,459bef486812aa25204be022145caa62,ES,1,6922.21,6922.21,2018-07-25 18:10:17,2018-07-25 18:10:17
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,6726.66,2017-05-24 18:14:34,2017-05-24 18:14:34
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,6081.54,2017-11-24 11:03:35,2017-11-24 11:03:35
7,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4764.34,2017-04-18 18:50:13,2017-04-18 18:50:13
8,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,4681.78,2018-06-22 12:23:19,2018-06-22 12:23:19
9,c8460e4251689ba205045f3ea17884a1,RS,4,4655.91,1163.98,2018-08-07 09:03:02,2018-08-08 14:27:15


## 2.5 Business Question 4: Seller Performance Ranking

> *"Which sellers drive the most revenue, and how do they perform on reviews?"*

In [6]:
q4 = run_query("""
    SELECT
        s.seller_id,
        s.seller_city,
        s.seller_state,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        ROUND(SUM(oi.price), 2) AS total_revenue,
        ROUND(AVG(r.review_score), 2) AS avg_review_score,
        COUNT(DISTINCT oi.product_id) AS unique_products
    FROM sellers s
    JOIN order_items oi ON s.seller_id = oi.seller_id
    JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY s.seller_id
    HAVING total_orders >= 10
    ORDER BY total_revenue DESC
    LIMIT 15
""")

print("Top 15 Sellers by Revenue (min 10 orders):")
display(q4)

Top 15 Sellers by Revenue (min 10 orders):


,seller_id,seller_city,seller_state,total_orders,total_revenue,avg_review_score,unique_products
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1124,226987.93,4.14,94
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,348,217940.44,4.13,23
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1772,199408.32,3.83,394
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,578,190917.14,4.37,284
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,973,188063.83,3.35,197
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,319,165981.49,4.36,175
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1311,162303.67,4.08,221
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1145,140238.65,4.27,146
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,910,139720.16,3.87,153
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1261,131906.71,4.09,103


## 2.6 Business Question 5: Delivery Performance by State

> *"Which states have the worst delivery times?"*

In [7]:
q5 = run_query("""
    SELECT
        c.customer_state AS state,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(AVG(
            JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_purchase_timestamp)
        ), 1) AS avg_delivery_days,
        ROUND(AVG(
            JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_estimated_delivery_date)
        ), 1) AS avg_delay_days,
        ROUND(100.0 * SUM(
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END
        ) / COUNT(*), 1) AS late_delivery_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
    ORDER BY avg_delivery_days DESC
""")

print("Delivery Performance by State:")
display(q5)

Delivery Performance by State:


,state,total_orders,avg_delivery_days,avg_delay_days,late_delivery_pct
0,RR,41,29.4,-16.6,12.2
1,AP,67,27.2,-19.1,4.5
2,AM,145,26.4,-18.9,4.1
3,AL,397,24.5,-8.0,23.9
4,PA,946,23.8,-13.4,12.4
5,MA,717,21.6,-8.9,19.7
6,SE,335,21.5,-9.3,15.2
7,CE,1279,21.3,-10.1,15.3
8,AC,80,21.0,-20.1,3.8
9,PB,517,20.4,-12.6,11.0


## 2.7 Business Question 6: Payment Behavior Analysis

> *"How do customers prefer to pay, and what does installment usage look like?"*

In [8]:
q6 = run_query("""
    SELECT
        p.payment_type,
        COUNT(*) AS transaction_count,
        ROUND(SUM(p.payment_value), 2) AS total_value,
        ROUND(AVG(p.payment_value), 2) AS avg_value,
        ROUND(AVG(p.payment_installments), 1) AS avg_installments
    FROM payments p
    JOIN orders o ON p.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.payment_type
    ORDER BY total_value DESC
""")

print("Payment Behavior Summary:")
display(q6)

Payment Behavior Summary:


,payment_type,transaction_count,total_value,avg_value,avg_installments
0,credit_card,74586,12101094.88,162.24,3.5
1,boleto,19191,2769932.58,144.33,1.0
2,voucher,5493,343013.19,62.45,1.0
3,debit_card,1486,208421.12,140.26,1.0


## 2.8 Business Question 7: Review Score vs Delivery Time

> *"Does delivery speed impact customer satisfaction?"*

In [9]:
q7 = run_query("""
    SELECT
        r.review_score,
        COUNT(*) AS num_reviews,
        ROUND(AVG(
            JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_purchase_timestamp)
        ), 1) AS avg_delivery_days,
        ROUND(AVG(
            JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_estimated_delivery_date)
        ), 1) AS avg_delay_days
    FROM reviews r
    JOIN orders o ON r.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY r.review_score
    ORDER BY r.review_score
""")

print("Review Score vs Delivery Time:")
display(q7)

Review Score vs Delivery Time:


,review_score,num_reviews,avg_delivery_days,avg_delay_days
0,1,9405,21.3,-3.4
1,2,2941,16.7,-7.9
2,3,7961,14.3,-10.1
3,4,18987,12.3,-11.7
4,5,57059,10.7,-12.7


## 2.9 Business Question 8: Repeat Customer Rate

> *"What percentage of customers have ordered more than once?"*

In [10]:
q8 = run_query("""
    WITH customer_orders AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT o.order_id) AS order_count
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id
    )
    SELECT
        CASE
            WHEN order_count = 1 THEN '1 order'
            WHEN order_count = 2 THEN '2 orders'
            WHEN order_count BETWEEN 3 AND 5 THEN '3-5 orders'
            ELSE '6+ orders'
        END AS order_bucket,
        COUNT(*) AS num_customers,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_orders), 2) AS pct_of_total
    FROM customer_orders
    GROUP BY order_bucket
    ORDER BY MIN(order_count)
""")

print("Repeat Customer Analysis:")
display(q8)

Repeat Customer Analysis:


,order_bucket,num_customers,pct_of_total
0,1 order,90557,97.00
1,2 orders,2573,2.76
2,3-5 orders,218,0.23
3,6+ orders,10,0.01


## 2.10 Business Question 9: Peak Sales Hours & Day of Week

> *"When do most orders come in?"*

In [11]:
q9_hour = run_query("""
    SELECT
        CAST(strftime('%H', order_purchase_timestamp) AS INTEGER) AS hour_of_day,
        COUNT(*) AS order_count,
        ROUND(SUM(p.payment_value), 2) AS revenue
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""")

q9_dow = run_query("""
    SELECT
        CASE CAST(strftime('%w', order_purchase_timestamp) AS INTEGER)
            WHEN 0 THEN 'Sunday'
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS day_of_week,
        COUNT(*) AS order_count,
        ROUND(SUM(p.payment_value), 2) AS revenue
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY CAST(strftime('%w', order_purchase_timestamp) AS INTEGER)
    ORDER BY CAST(strftime('%w', order_purchase_timestamp) AS INTEGER)
""")

print("Orders by Hour of Day:")
display(q9_hour)
print("\nOrders by Day of Week:")
display(q9_dow)

Orders by Hour of Day:


,hour_of_day,order_count,revenue
0,0,2464,360018.23
1,1,1178,171604.47
2,2,518,64572.21
3,3,267,37551.29
4,4,212,28209.97
5,5,189,25620.74
6,6,490,63569.25
7,7,1236,173948.18
8,8,3020,452537.94
9,9,4814,775133.99



Orders by Day of Week:


,day_of_week,order_count,revenue
0,Sunday,12094,1808035.08
1,Monday,16368,2530591.86
2,Tuesday,16222,2474065.60
3,Wednesday,15760,2396624.55
4,Thursday,14977,2284158.44
5,Friday,14315,2222878.71
6,Saturday,11020,1706107.53


## 2.11 Business Question 10: Revenue by Customer State

> *"Which states generate the most revenue per capita?"*

In [12]:
q10 = run_query("""
    SELECT
        c.customer_state AS state,
        COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(p.payment_value), 2) AS total_revenue,
        ROUND(SUM(p.payment_value) / COUNT(DISTINCT c.customer_unique_id), 2) AS revenue_per_customer
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    ORDER BY total_revenue DESC
""")

print("Revenue by Customer State:")
display(q10)

Revenue by Customer State:


,state,unique_customers,total_orders,total_revenue,revenue_per_customer
0,SP,39155,40500,5770266.19,147.37
1,RJ,11917,12350,2055690.45,172.50
2,MG,11001,11354,1819277.61,165.37
3,RS,5168,5345,861802.40,166.76
4,PR,4769,4923,781919.55,163.96
5,SC,3449,3546,595208.40,172.57
6,BA,3158,3256,591270.60,187.23
7,DF,2019,2080,346146.17,171.44
8,GO,1895,1957,334294.22,176.41
9,ES,1928,1995,317682.65,164.77


In [13]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.


## 2.12 Key SQL Insights

| Query | Key Finding |
|---|---|
| **Monthly Revenue** | Strong growth from 2017 to mid-2018, with seasonal peaks |
| **Top Categories** | Health & beauty, watches, and bed/bath/table dominate revenue |
| **Customer LTV** | Most customers are one-time buyers; repeat rate is very low |
| **Seller Performance** | Top sellers concentrate revenue; review scores vary |
| **Delivery by State** | Northern states experience significantly longer delivery times |
| **Payment Behavior** | Credit card (74%); average 3-4 installments |
| **Reviews vs Delivery** | Strong inverse correlation — faster delivery = higher scores |
| **Repeat Rate** | ~97% of customers placed only 1 order — retention is a major gap |
| **Peak Hours** | Orders peak between 10 AM – 4 PM; Monday is the busiest day |
| **Revenue by State** | SP dominates in absolute revenue; smaller states have higher per-customer spend |

---

*Next: Module 3 — Visualizations, RFM Segmentation & Dashboard →*